# Where grep must win, and where it must lose

Before comparing an agent that greps with a dense retriever on a benchmark, build the questions where grep has to win if it works at all, and the questions where it has to lose. This notebook runs `C5` (`grep` and `open` only) and `C0` (dense `search` only) on four such sets, on the on-prem `Qwen3_6-35B-A3B` with thinking off and a 30-call budget.

The most important result on this page is a control that failed. The paraphrase control was built so that grep would lose: 50 rewrites with no capitalized span of their supporting passages. `C5` wins it anyway, F1 0.514 against `C0`'s 0.304, because the model names the entity from its own knowledge and greps the name. In more than half of the 50 questions, the first grep that matched used a term the question does not contain, such as "The Book Thief" for "the novel about a girl who steals books"; the paraphrase section counts them. On Wikipedia-derived questions, then, `C5`'s edge rests partly on the model knowing the entities, and a paraphrase does not take literal access away. For the control to work, the entities must be ones the model cannot know; the cheapest test renames every entity in the 50 rewrites and their passages to consistent invented names and reruns both arms.

The existence proof does not separate the arms on this model. On the 75 four-hop questions `C5` reaches F1 0.457 against `C0`'s 0.434, a paired difference of +0.022 (95% bootstrap interval -0.054 to +0.104), and sees every supporting passage of 29% of them, against the probe's literal-grep ceiling of 43%; RI #4's `dense` has every supporting passage in its top 10 for none. `C5` gets there mostly by reading bridge entities off `grep` previews: it opens every supporting passage of only 4% of these questions. `C0`, the agent loop with dense search alone, closes most of the gap by searching again with reworded queries. For `C5` to win here, `C0`'s rewording would have to stop finding the later hops, and the 139,416-passage pool in notebook 03 is the cheapest test.

On ServerFault error strings, `C5` opens the gold post for all 30 questions at 4,200 tokens per question; `C0` surfaces it for all 30 and reads it in full for 19, at 17,900. On the near-miss nulls, neither arm abstains often: `C5` answers "unknown" for 14% and `C0` for 6%, so an empty grep did not become an abstention here, as the set's own statistics predicted.

The four sets:

- 4-hop MuSiQue bridge questions, the existence proof. Dense retrieval never returns every supporting passage of these 75 questions in its top 10 (RI #4's `dense` row), and each hop's answer is an entity named in the next passage, which a literal search can follow.
- The paraphrase control: MuSiQue questions rewritten so that no capitalized span of a supporting passage appears in them, kept only when a model answers the rewrite from its supporting passages alone. Grep has nothing literal to match, so `C5` must lose here.
- Near-miss null questions: one name in an answerable question changed so that the corpus no longer answers it. An empty grep is a candidate signal to abstain; dense search returns the near miss either way.
- ServerFault error strings: 30 questions keyed on an exact error message, code, or identifier that appears in one post of the 10,000-question slice.

With `RUN_EXPENSIVE = False` the notebook reads the reference trajectories from the cache. With it on, the last cell runs 10 questions per panel on your model.

MuSiQue (Trivedi, Balasubramanian, Khot, and Sabharwal, TACL 2022) is licensed CC BY 4.0. ServerFault posts are licensed CC BY-SA; `CORPUS.md` records the versions.

In [1]:
import json
import os
import re
from pathlib import Path

from agent_arms import (
    SERVERFAULT_DESCRIPTION,
    abstained,
    dense_search,
    live_questions,
    musique_corpus,
    musique_set,
    run_arm,
    serverfault_set,
)
from agent_report import public_row, summarise
from aieng.advanced_rag import spend
from aieng.advanced_rag.corpora.musique import load_musique, stratified_test_set
from aieng.advanced_rag.models import choose_models


os.environ.setdefault("HF_HUB_OFFLINE", "1")

# The live cell calls your model. False reads the reference trajectories.
RUN_EXPENSIVE = False
MODELS = choose_models(show=True)

_, all_questions = load_musique()
SHARED = stratified_test_set(all_questions)
FOUR_HOP = [q for q in SHARED if q.hops == 4]
corpus = musique_corpus(11_656)
RI4 = json.loads(Path("../graphrag_technical_support/outputs/musique_graph_test.json").read_text())
print(f"{len(FOUR_HOP)} four-hop questions over {len(corpus):,} passages")

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget not set on the key; assuming $300.00 (set ADVANCED_RAG_KEY_BUDGET_USD to override), spent $135.44, left $164.56


75 four-hop questions over 11,656 passages


## How far a literal grep can reach

A model-free probe, from `outputs/musique_grep_probe.json` (`python grep_probe.py` rebuilds it). First hop: some capitalized span of the question matches the first supporting passage. Bridge: the previous hop's gold answer matches the next supporting passage. Chain: every hop does. The chain column is a ceiling for `C5`, because it hands the agent the right bridge answers; it is not an accuracy.

In [2]:
probe = json.loads(Path("outputs/musique_grep_probe.json").read_text())["shared-11656"]
dense = RI4["arms"]["dense"]["summary"]
print(f"{'hops':>4} {'n':>4} {'first hop':>10} {'bridge':>7} {'chain':>6}   dense all-supporting@10")
for hops, row in probe["by_hop"].items():
    print(
        f"{hops:>4} {row['questions']:>4} {row['first_hop']:>10.3f} {row['bridge']:>7.3f} {row['chain']:>6.3f}   {dense[hops + 'hop']['all_supporting_recall_at_10']:.3f}"
    )
sizes = probe["hit_sizes"]
print(f"passages per bridge grep: median {sizes['bridge_median']}, 90th percentile {sizes['bridge_p90']}")

hops    n  first hop  bridge  chain   dense all-supporting@10
   2  150      0.833   0.933  0.787   0.387
   3   75      0.867   0.827  0.653   0.173
   4   75      0.733   0.804  0.427   0.000
passages per bridge grep: median 12, 90th percentile 87


## The existence proof: `C5` against `C0` on 4-hop questions

All-supporting recall here is over the passages each run read in full. `seen, not read` is the share of questions where a supporting passage appeared in a result and was never opened.

In [3]:
gold = {q.question_id: q.supporting_ids for q in FOUR_HOP}
four = {}
for arm in ("C0", "C5", "grep-iterative"):
    value = run_arm(arm, FOUR_HOP, corpus=corpus, label="4hop-11656", models=MODELS, live=False).value
    four[arm] = summarise([public_row(r) for r in value["runs"]], gold)["all"]
print()
print(
    f"{'arm':<16}{'recall':>7}{'seen':>7}{'seen, not read':>16}{'EM':>7}{'F1':>7}{'calls/q':>9}{'tokens/q':>10}{'exhaust':>9}"
)
for arm, s in four.items():
    print(
        f"{arm:<16}{s['all_supporting_recall_read']:>7.3f}{s['all_supporting_recall_seen']:>7.3f}{s['seen_not_read_rate']:>16.3f}"
        f"{s['exact_match']:>7.3f}{s['f1']:>7.3f}{s['calls_per_question']:>9.1f}{s['tokens_per_question']:>10,.0f}{s['exhaustion_rate']:>9.2f}"
    )
for arm in ("dense", "dense-iterative", "hipporag2"):
    s = RI4["arms"][arm]["summary"]["4hop"]
    print(
        f"{arm + ' (RI #4)':<16}{s['all_supporting_recall_at_10']:>7.3f}{'':>7}{'':>16}{s['exact_match']:>7.3f}{s['f1']:>7.3f}   (recall is top 10 ranked)"
    )
print(f"literal-grep chain ceiling at 4 hops: {probe['by_hop']['4']['chain']:.3f}")

agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           792 calls, 6,516,936 prompt + 126,730 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 792 calls, est. $2.27 (set RUN_EXPENSIVE = True to rebuild)
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           1,306 calls, 5,470,097 prompt + 157,722 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 1,306 calls, est. $2.04 (set RUN_EXPENSIVE = True to rebuild)
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           419 calls, 209,404 prompt + 5,407 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 419 calls, est. $0.08 (set RUN_EXPENSIVE = True to rebuild)

arm              recall   seen  seen, not read     EM     F1  calls/q  tokens/q  exhaust
C0                0.240  0.240           0.000  0.293  0.434      9.6    88,582     0.09
C5                0.

## The paraphrase control

Rewrites with no literal span of their supporting passages. Grep should lose here, and if it does not, the 4-hop result above says less about literal matching than it seems to.

In [4]:
paraphrase = musique_set("paraphrase")
para_gold = {q.question_id: q.supporting_ids for q in paraphrase}
para = {}
for arm in ("C0", "C5"):
    value = run_arm(arm, paraphrase, corpus=corpus, label="paraphrase-11656", models=MODELS, live=False).value
    para[arm] = summarise([public_row(r) for r in value["runs"]], para_gold)["all"]
print(f"{len(paraphrase)} rewrites: " + ", ".join(f"{h}-hop {sum(q.hops == h for q in paraphrase)}" for h in (2, 3, 4)))
print(f"{'arm':<5}{'F1':>7}{'EM':>7}{'read':>7}{'seen':>7}{'calls/q':>9}{'tokens/q':>10}")
for arm, s in para.items():
    print(
        f"{arm:<5}{s['f1']:>7.3f}{s['exact_match']:>7.3f}{s['all_supporting_recall_read']:>7.3f}{s['all_supporting_recall_seen']:>7.3f}{s['calls_per_question']:>9.1f}{s['tokens_per_question']:>10,.0f}"
    )

# Each C5 run's first matching grep, and whether the question contains its pattern.
c5_runs = run_arm(
    "C5", paraphrase, corpus=corpus, label="paraphrase-11656", models=MODELS, live=False, show=False
).value["runs"]
asked = {q.question_id: q.question.lower() for q in paraphrase}
first_hits = {}
for run in c5_runs:
    hit = next((e for e in run["trace"] if e["name"] == "grep" and (e.get("total_hits") or 0) > 0), None)
    if hit is not None:
        first_hits[run["question_id"]] = re.sub(r"\\(.)", r"\1", str(hit["arguments"].get("pattern", "")))
outside = [qid for qid, pattern in first_hits.items() if pattern.lower() not in asked[qid]]
print()
print(
    f"C5's first matching grep used a pattern the question does not contain in {len(outside)} of "
    f"{len(paraphrase)} questions ({len(first_hits)} had a matching grep)"
)
for qid in outside[:3]:
    print(f"  {first_hits[qid]!r}")

agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           294 calls, 1,540,612 prompt + 74,185 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 294 calls, est. $0.65 (set RUN_EXPENSIVE = True to rebuild)
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           847 calls, 3,357,677 prompt + 106,403 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 847 calls, est. $1.27 (set RUN_EXPENSIVE = True to rebuild)
50 rewrites: 2-hop 0, 3-hop 29, 4-hop 21
arm       F1     EM   read   seen  calls/q  tokens/q
C0     0.304  0.200  0.160  0.160      4.9    32,296
C5     0.514  0.380  0.040  0.220     15.9    69,282

C5's first matching grep used a pattern the question does not contain in 29 of 50 questions (46 had a matching grep)
  '1999 Major League Baseball draft'
  '1868.*Cardinal'
  'Pete Rose'


## Near-miss null questions

The corpus does not answer these. A correct run answers "unknown"; the abstention rate is the share that do. `zero-hit greps` is the share of `C5`'s greps that matched nothing, the signal the hypothesis says should lead to abstaining.

This set is a weak test of that signal, and the numbers say why before any run. Only 17 of the 50 changes introduce a name or year the original question lacked, and a literal grep for the new term finds nothing in the 11,656 passages for 4 of those 17 (and for 1 in the 139,416-passage pool). The other changes swap words that already occur in the corpus, so an empty grep cannot flag them. The nulls were checked as unanswerable against the 11,656 passages only, so this section runs on that corpus.

In [5]:
nulls = musique_set("nulls")
print(f"{len(nulls)} near-miss null questions")
print(f"{'arm':<5}{'abstained':>10}{'calls/q':>9}{'tokens/q':>10}{'zero-hit greps':>16}")
for arm in ("C0", "C5"):
    runs = run_arm(arm, nulls, corpus=corpus, label="nulls-11656", models=MODELS, live=False, show=False).value["runs"]
    rate = sum(abstained(r["answer"]) for r in runs) / len(runs)
    calls = sum(r["model_calls"] for r in runs) / len(runs)
    tokens = sum(r["prompt_tokens"] + r["completion_tokens"] for r in runs) / len(runs)
    greps = [e for r in runs for e in r["trace"] if e["name"] == "grep"]
    zero = f"{sum(e['total_hits'] == 0 for e in greps) / len(greps):.2f}" if greps else "-"
    print(f"{arm:<5}{rate:>10.2f}{calls:>9.1f}{tokens:>10,.0f}{zero:>16}")

50 near-miss null questions
arm   abstained  calls/q  tokens/q  zero-hit greps
C0         0.06     12.9   153,065               -
C5         0.14     14.3    58,651            0.56


## ServerFault error strings

Each question quotes an exact error string from one post. The score is whether the run read the gold post. Dense search previews a post's first 512 tokens; a post read in full counts only when the agent opens it or search returns it whole.

In [6]:
sf_corpus, sf_questions = serverfault_set()
sf_gold = {q.question_id: set(q.supporting_ids) for q in sf_questions}
print(f"{len(sf_corpus):,} posts, {len(sf_questions)} questions")
sf = {}
for arm in ("C0", "C5"):
    sf[arm] = run_arm(
        arm,
        sf_questions,
        corpus=sf_corpus,
        label="errors-serverfault",
        models=MODELS,
        live=False,
        description=SERVERFAULT_DESCRIPTION,
    ).value
print()
print(f"{'arm':<5}{'gold read':>10}{'gold seen':>10}{'calls/q':>9}{'tokens/q':>10}{'exhaust':>9}")
for arm, value in sf.items():
    runs = value["runs"]
    n = len(runs)
    read = sum(bool(sf_gold[r["question_id"]] & set(r["read_ids"])) for r in runs) / n
    seen = sum(bool(sf_gold[r["question_id"]] & (set(r["read_ids"]) | set(r["seen_ids"]))) for r in runs) / n
    calls = sum(r["model_calls"] for r in runs) / n
    tokens = sum(r["prompt_tokens"] + r["completion_tokens"] for r in runs) / n
    exhausted = sum(bool(r["exhausted"]) for r in runs) / n
    print(f"{arm:<5}{read:>10.3f}{seen:>10.3f}{calls:>9.1f}{tokens:>10,.0f}{exhausted:>9.2f}")

9,999 posts, 30 questions
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           102 calls, 530,685 prompt + 6,866 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 102 calls, est. $0.18 (set RUN_EXPENSIVE = True to rebuild)
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           92 calls, 121,671 prompt + 4,076 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 92 calls, est. $0.05 (set RUN_EXPENSIVE = True to rebuild)

arm   gold read gold seen  calls/q  tokens/q  exhaust
C0        0.633     1.000      3.4    17,918     0.00
C5        1.000     1.000      3.1     4,192     0.00


## Run 10 questions per panel on your model

With `RUN_EXPENSIVE = True`: `C0` and `C5` on 10 of the 4-hop questions and 10 of the ServerFault questions, on your generation model. Compare your two arms with each other.

In [7]:
if RUN_EXPENSIVE:
    panels = {
        "4-hop": (live_questions(FOUR_HOP, "claude-opus-5-5"), corpus, "4hop-live", None),
        "ServerFault": (sf_questions[:10], sf_corpus, "errors-live", SERVERFAULT_DESCRIPTION),
    }
    for panel, (questions, panel_corpus, label, description) in panels.items():
        ids = {q.question_id: set(q.supporting_ids) for q in questions}
        for arm in ("C0", "C5"):
            extra = {"description": description} if description else {}
            value = run_arm(
                arm,
                questions,
                corpus=panel_corpus,
                label=label,
                models=MODELS,
                live=True,
                search=lambda c=panel_corpus: dense_search(c),
                **extra,
            ).value
            runs = value["runs"]
            read = sum(ids[r["question_id"]] <= set(r["read_ids"]) for r in runs) / len(runs)
            print(
                f"{panel:<12} {arm} on {value['model']}: all gold read {read:.2f}, {sum(r['model_calls'] for r in runs) / len(runs):.1f} calls/q"
            )
else:
    print("Set RUN_EXPENSIVE = True to run C0 and C5 on your model.")

Set RUN_EXPENSIVE = True to run C0 and C5 on your model.


In [8]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
this run, proxy usage $0.00 (team key: includes teammates' calls in the same window)
key budget not set on the key; assuming $300.00 (set ADVANCED_RAG_KEY_BUDGET_USD to override), spent $135.44, left $164.56
